# Task 5A: Characteristics of Evaluation Metrics

This notebook defines the MSE and MAE evaluation metrics, discusses when each is preferable, and demonstrates a special case where the two metrics yield identical values.

## 1. Formulae

**Mean Squared Error (MSE):**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Mean Absolute Error (MAE):**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Where:
- $n$ is the number of observations
- $y_i$ is the true value for observation $i$
- $\hat{y}_i$ is the predicted value for observation $i$

Both metrics are **non-negative** and equal zero only when predictions are perfect. The key difference lies in the exponent: MSE squares each error, while MAE takes the absolute value.

## 2. Discussion: When to Use Which

### When MSE is preferred

- **Large errors are disproportionately costly.** Because MSE squares each residual, an error of 2 contributes 4 to the sum while an error of 1 contributes only 1. This makes MSE especially suitable in domains where big misses are far worse than small ones -- for example, medical dosage prediction or financial risk modelling.
- **Smooth optimization landscape.** MSE is differentiable everywhere with respect to the predictions, which makes gradient-based optimization (e.g., gradient descent in neural networks) straightforward.
- **In our mood prediction context:** a prediction that is off by 2 points incurs 4x the penalty of being off by 1 point. This is desirable because a large mood prediction error -- say, predicting a patient feels fine when they are actually in crisis -- is clinically much more dangerous than a small miss.

### When MAE is preferred

- **Robustness to outliers.** MAE treats all errors linearly, so a single extreme outlier does not dominate the metric. This makes MAE more appropriate when the data contains noisy or unreliable extreme values.
- **Median-like estimates.** Minimising MAE during training produces predictions that converge toward the conditional *median* of the target distribution, whereas minimising MSE converges toward the conditional *mean*. If the target distribution is skewed, the median can be a more representative central tendency.
- **Interpretability.** MAE has a direct interpretation: "on average, our prediction is off by X units." In our context, an MAE of 0.4 means the model is typically wrong by 0.4 mood points on the 1--10 scale, which is immediately understandable to clinicians.

### When MAE is preferred (cont.)

- **Non-differentiable at zero.** MAE has a kink at $|e_i| = 0$, which can cause issues for gradient-based optimizers (sub-gradient methods are needed). In practice, libraries handle this automatically, but it can slow convergence.

### Summary table

| Property | MSE | MAE |
|---|---|---|
| Error weighting | Quadratic (large errors penalised more) | Linear (all errors equal weight) |
| Sensitivity to outliers | High | Low |
| Optimal predictor | Conditional mean | Conditional median |
| Differentiability | Smooth everywhere | Kink at 0 |
| Interpretability | Less intuitive (squared units) | Direct ("average error in original units") |

## 3. Example: When MSE Equals MAE

### Mathematical argument

Consider the case where **every** absolute residual has the same value $c$:

$$|y_i - \hat{y}_i| = c \quad \forall \, i$$

Then:

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} c = c$$

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} c^2 = c^2$$

Setting $\text{MSE} = \text{MAE}$:

$$c^2 = c \implies c(c - 1) = 0 \implies c = 0 \text{ or } c = 1$$

So MSE and MAE are equal when **all absolute errors are exactly 0** (trivial -- both metrics are zero) **or when all absolute errors are exactly 1**.

This is the only non-trivial scenario for constant-error predictions: every prediction misses by exactly 1 unit, giving $\text{MSE} = \text{MAE} = 1$.

Below we verify this numerically and contrast it with varied errors where MSE > MAE.

In [1]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --- Case 1: All errors equal 1 (MSE = MAE) ---
y_true_1 = np.array([3.0, 5.0, 7.0, 2.0, 8.0])
y_pred_1 = np.array([4.0, 6.0, 8.0, 3.0, 9.0])  # every prediction is off by exactly 1

errors_1 = np.abs(y_true_1 - y_pred_1)
mse_1 = mean_squared_error(y_true_1, y_pred_1)
mae_1 = mean_absolute_error(y_true_1, y_pred_1)

print("Case 1: All absolute errors = 1")
print(f"  Absolute errors: {errors_1}")
print(f"  MSE = {mse_1:.4f}")
print(f"  MAE = {mae_1:.4f}")
print(f"  MSE == MAE? {np.isclose(mse_1, mae_1)}")

print()

# --- Case 2: Varied errors (MSE > MAE) ---
y_true_2 = np.array([3.0, 5.0, 7.0, 2.0, 8.0])
y_pred_2 = np.array([3.5, 4.0, 7.0, 4.0, 7.5])  # errors: 0.5, 1.0, 0.0, 2.0, 0.5

errors_2 = np.abs(y_true_2 - y_pred_2)
mse_2 = mean_squared_error(y_true_2, y_pred_2)
mae_2 = mean_absolute_error(y_true_2, y_pred_2)

print("Case 2: Varied absolute errors")
print(f"  Absolute errors: {errors_2}")
print(f"  MSE = {mse_2:.4f}")
print(f"  MAE = {mae_2:.4f}")
print(f"  MSE > MAE? {mse_2 > mae_2}")

print()

# --- Case 3: All errors equal 0 (trivial) ---
y_true_3 = np.array([3.0, 5.0, 7.0])
y_pred_3 = np.array([3.0, 5.0, 7.0])

mse_3 = mean_squared_error(y_true_3, y_pred_3)
mae_3 = mean_absolute_error(y_true_3, y_pred_3)

print("Case 3: Perfect predictions (trivial)")
print(f"  MSE = {mse_3:.4f}")
print(f"  MAE = {mae_3:.4f}")
print(f"  MSE == MAE? {np.isclose(mse_3, mae_3)}")

Case 1: All absolute errors = 1
  Absolute errors: [1. 1. 1. 1. 1.]
  MSE = 1.0000
  MAE = 1.0000
  MSE == MAE? True

Case 2: Varied absolute errors
  Absolute errors: [0.5 1.  0.  2.  0.5]
  MSE = 1.1000
  MAE = 0.8000
  MSE > MAE? True

Case 3: Perfect predictions (trivial)
  MSE = 0.0000
  MAE = 0.0000
  MSE == MAE? True


### Interpretation

The numerical results confirm the mathematical derivation:

- **Case 1** (constant error = 1): MSE = MAE = 1.0. When every prediction is off by exactly 1 unit, squaring preserves the value ($1^2 = 1$), so both metrics agree.
- **Case 2** (varied errors): MSE > MAE. This is the typical situation -- because MSE squares the errors, the larger residuals (e.g., the error of 2.0) receive disproportionate weight, inflating MSE relative to MAE.
- **Case 3** (perfect predictions): MSE = MAE = 0. The trivial case where no error exists.

The key insight is that MSE = MAE is a **degenerate case** that only arises under very specific conditions (all absolute errors equal to 0 or 1). In any realistic prediction scenario with varied error magnitudes, MSE will exceed MAE because the squaring operation amplifies larger errors.